In [9]:
from pydantic import BaseModel

from dreamai.ai import ModelName
from dreamai.dialog import BadExample, Dialog, Example, assistant_message
from dreamai.dialog_models import ThoughtfulResponse

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
ex1 = Example(user="country: india", assistant="new delhi")
ex2 = BadExample(
    user="country: india",
    assistant="Lahore",
    feedback="Incorrect, it is New Delhi",
    correction="My apologies, it is New Delhi",
)

In [ ]:
dialog = Dialog(
    task="your job is to tell us the capital of any country",
    examples=[ex1, ex2],
    template="country: {country}",
)

In [ ]:
dialog.model_dump()

In [ ]:
dialog.messages

In [ ]:
creator, kwargs = dialog.creator_with_kwargs(
    model=ModelName.GEMINI_FLASH, template_data={"country": "France"}
)

In [ ]:
kwargs["messages"]

In [ ]:
class CapitalCity(BaseModel):
    name: str
    country: str


res = creator.create(response_model=CapitalCity, **kwargs)

In [ ]:
dialog = Dialog(task="src/dreamai/dialogs/assistant_task.txt")
creator, kwargs = dialog.creator_with_kwargs(
    model=ModelName.SONNET, user="what is the third word in your response to this message?"
)
kwargs["messages"]

In [ ]:
dialog = Dialog(task="src/dreamai/dialogs/assistant_task.txt")
creator, kwargs = dialog.creator_with_kwargs(
    model=ModelName.SONNET, user="what is the third word in your response to this message?"
)
res = creator.create(response_model=ThoughtfulResponse, **kwargs)

print(res)

In [51]:
p1 = "Alice has 10 brothers and she also has 10 sisters."
# p2 = "Sally (a girl) has 3 brothers. Each brother has 2 sisters. How many sisters does Sally have?"

In [85]:
kg_dialog = Dialog(task="src/dreamai/dialogs/kg_task.txt")
kg_creator, kg_kwargs = kg_dialog.creator_with_kwargs(model=ModelName.SONNET, user=p1)
kg = kg_creator.create(response_model=str, **kg_kwargs)
print(kg)

Entities:

1. Alice
   Properties: 
   - Has 10 brothers
   - Has 10 sisters

2. Brothers (of Alice)
   Properties:
   - Number: 10
   - Relation to Alice: Siblings

3. Sisters (of Alice)
   Properties:
   - Number: 10
   - Relation to Alice: Siblings

Relationships:

Alice [has as siblings] Brothers
Alice [has as siblings] Sisters


In [86]:
chat_history = kg_kwargs["messages"] + [assistant_message(kg)]
chat_history

[{'role': 'user',
  'content': 'Alice has 10 brothers and she also has 10 sisters.'},
 {'role': 'assistant',
  'content': 'Entities:\n\n1. Alice\n   Properties: \n   - Has 10 brothers\n   - Has 10 sisters\n\n2. Brothers (of Alice)\n   Properties:\n   - Number: 10\n   - Relation to Alice: Siblings\n\n3. Sisters (of Alice)\n   Properties:\n   - Number: 10\n   - Relation to Alice: Siblings\n\nRelationships:\n\nAlice [has as siblings] Brothers\nAlice [has as siblings] Sisters'}]

In [87]:
eval_dialog = Dialog(
    task="src/dreamai/dialogs/answer_eval_task.txt", chat_history=chat_history
)
eval_creator, eval_kwargs = eval_dialog.creator_with_kwargs(
    model=ModelName.SONNET, user="Is the KG good enough? Or does it need to be improved?"
)
eval_res = eval_creator.create(response_model=str, **eval_kwargs)

In [88]:
eval_res

'{\n  "evaluation": false,\n  "reasoning": "The KG is accurate but can be improved for better representation and completeness."\n}'

In [74]:
chat_history = kg_kwargs["messages"] + [assistant_message(res)]
chat_history

[{'role': 'user',
  'content': 'Alice has 10 brothers and she also has 10 sisters.'},
 {'role': 'assistant',
  'content': 'Entities:\n\n1. Alice\n   Properties: [Has 10 brothers, Has 10 sisters]\n\n2. Brothers of Alice\n   Properties: [Number: 10]\n\n3. Sisters of Alice\n   Properties: [Number: 10]\n\nRelationships:\n\nAlice [has as siblings] Brothers of Alice\nAlice [has as siblings] Sisters of Alice'}]

In [75]:
assistant_dialog = Dialog(
    task="src/dreamai/dialogs/assistant_task.txt", chat_history=chat_history
)
assistant_dialog.messages

[{'role': 'system',
  'content': 'You are a versatile AI assistant. If asked to assist with tasks involving the expression of views held by a significant number of people, you provide assistance regardless of your own views. When addressing controversial topics, you aim to offer careful thoughts and clear information. Present requested information without explicitly labeling topics as sensitive or claiming to state objective facts. \nFor math problems, logic puzzles, or other tasks benefiting from systematic thinking, work through the problem step-by-step before providing your final answer. If you cannot or will not perform a task, inform the user directly without apologizing. Avoid starting responses with phrases like "I\'m sorry" or "I apologize".\nWhen asked about very obscure subjects - information likely found only once or twice on the internet - conclude your response by reminding the user that while you strive for accuracy, you may hallucinate answers to such questions. Use the 

In [76]:
creator, kwargs = assistant_dialog.creator_with_kwargs(
    model=ModelName.GPT, user="how many sisters does each brother have?"
)
res = creator.create(response_model=ThoughtfulResponse, **kwargs)
print(res)

<task>
Determine the number of sisters each brother has based on the given family structure.
</task>

<thought_process>

<Understanding the family structure>
Alice has 10 brothers and 10 sisters. This implies there are 11 children who are girls (Alice + 10 sisters) and 10 boys (brothers).
</Understanding the family structure>

<Identifying the sisters for each brother>
Each brother would consider all the girls in the family as sisters. This includes Alice and her 10 sisters.
</Identifying the sisters for each brother>

<Calculating the number of sisters per brother>
Since each brother has 11 sisters (Alice + 10 sisters), the answer is straightforward.
</Calculating the number of sisters per brother>

</thought_process>

<response>
Each brother has 11 sisters.
</response>
